In [4]:
import sys , os 
import geopandas as gpd 
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
data_path = os.path.join(os.getcwd(),'..','data','geojson','polygon_cleaned.geojson')
gdf = gpd.read_file(data_path)

In [5]:
gdf

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,url_type,download_url,poly_id,image_uid,geometry
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,0,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ..."
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ..."
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,2,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ..."
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,3,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ..."
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,4,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,1528.937,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1948,img_0029_37df798b,"MULTIPOLYGON (((-75.123 3.80901, -75.12297 3.8..."
1949,333,Bare areas,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,0.026,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1949,img_0029_37df798b,"MULTIPOLYGON (((-75.12391 3.81075, -75.1239 3...."
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1950,img_0029_37df798b,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3...."
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1951,img_0029_37df798b,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3..."


In [6]:
base_url=os.path.join(os.getcwd(),'..','data','images','cog')
row = gdf.iloc[0]
cog_url = f"{base_url}/{row['image_uid']}.tif"
geom = [row.geometry.__geo_interface__]
cog_url


'/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/cog/img_0000_c4a77f3e.tif'

In [7]:
import rasterio 
from rasterio.mask import mask
import numpy as np 
with rasterio.open(cog_url) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    
    print("\nOriginal geometry CRS:", data.crs)
    print("Original geometry bounds:", row.geometry.bounds)
    
    geom_gdf = gpd.GeoDataFrame([row], geometry='geometry', crs=data.crs)
    geom_reprojected = geom_gdf.to_crs(src.crs)
    geom_transformed = [geom_reprojected.geometry.iloc[0].__geo_interface__]
    
    print("\nTransformed bounds:", geom_reprojected.geometry.iloc[0].bounds)
    
    masked_data, _ = mask(src, geom_transformed, crop=True, all_touched=False)

    stats = {}
    for band_idx, band_name in enumerate(['r', 'g', 'b'], start=1):
        band_data = masked_data[band_idx - 1]
        
        is_masked = np.ma.isMaskedArray(band_data) 

        if is_masked:
            valid = band_data[~band_data.mask] # data=[1,2,0,4], mask=[false,false,true,false] -> valid = [1,2,4]
        else:
            nodata_mask = band_data != src.nodata # plain ndarray: [1, 2, -9999, 4], nodata = -9999 -> valid = [1,2,4]
            valid = band_data[nodata_mask]

        # we need to remove the noise here , may be more roboust ? ---- 
        
        if valid.size > 0:
            stats[f'{band_name}_mean'] = float(np.mean(valid))
            stats[f'{band_name}_std'] = float(np.std(valid))
        else:
            stats[f'{band_name}_mean'] = np.nan
            stats[f'{band_name}_std'] = np.nan
    print(stats)

Raster CRS: EPSG:32618
Raster bounds: BoundingBox(left=721782.2054845318, bottom=1174560.63571348, right=722310.3216997313, top=1174796.2209395552)

Original geometry CRS: EPSG:4326
Original geometry bounds: (-72.9714814422654, 10.618840012053344, -72.96914035652021, 10.620975138959965)

Transformed bounds: (721925.7615761297, 1174560.7526012938, 722182.7255575805, 1174796.086679114)
{'r_mean': 67.11867219203522, 'r_std': 63.05077827901257, 'g_mean': 72.36945632182152, 'g_std': 65.61500049739803, 'b_mean': 50.0079449993589, 'b_std': 48.51453627072589}
